# 02. Разметка статей по компаниям
**Вход:** `scraped_articles.csv` (выход ноутбука 01)  
**Выход:** `articles_labeled_long.csv` — long format, одна строка = статья × тикер  

Правила разметки:
- ≥ 2 упоминаний в теле статьи **или** ≥ 1 в заголовке → статья помечается тикером
- GAZP/SIBN: сначала проверяем «газпром нефть» (SIBN), потом «газпром» (GAZP)
- Одна статья может быть помечена несколькими тикерами

In [ ]:
# ── ЯЧЕЙКА 1: Настройки ──────────────────────────────────────
import pandas as pd
import re

INPUT_FILE  = '/content/scraped_articles.csv'
OUTPUT_FILE = '/content/articles_labeled_long.csv'

MIN_BODY_MENTIONS  = 2
MIN_TITLE_MENTIONS = 1

In [ ]:
# ── ЯЧЕЙКА 2: Словарь компаний ───────────────────────────────
# Ключ — тикер, значение — список паттернов
# ВАЖНО: SIBN должен идти ДО GAZP в словаре

COMPANY_KEYWORDS = {
    # ── Нефть и газ ──────────────────────────────────────────
    'LKOH':  ['лукойл', 'lukoil'],
    'SIBN':  ['газпром нефть', 'газпромнефть', 'gazprom neft'],
    'GAZP':  ['газпром', 'gazprom'],
    'ROSN':  ['роснефть', 'rosneft'],
    'NVTK':  ['новатэк', 'novatek'],
    'SNGS':  ['сургутнефтегаз'],
    'TRNFP': ['транснефть'],
    'BANE':  ['башнефть'],
    'RASP':  ['распадская'],
    # ── Металлургия и горнодобыча ────────────────────────────
    'GMKN':  ['норникель', 'норильский никель', 'norilsk nickel'],
    'NLMK':  ['нлмк'],
    'CHMF':  ['северсталь', 'severstal'],
    'MAGN':  ['ммк', 'магнитогорский металлургический'],
    'ALRS':  ['алроса'],
    'PLZL':  ['полюс'],
    'RUAL':  ['русал', 'rusal'],
    'ENPG':  ['эн+', 'en+'],
    'UGLD':  ['южуралзолото', 'югк'],
    'MTLR':  ['мечел'],
    # ── Банки и финансы ──────────────────────────────────────
    'SBER':  ['сбербанк', 'сбер'],
    'VTBR':  ['втб'],
    'CBOM':  ['мкб', 'московский кредитный банк'],
    'SVCB':  ['совкомбанк'],
    'BSPB':  ['банк санкт-петербург', 'банк санкт петербург', 'бсп'],
    'MOEX':  ['мосбиржа', 'московская биржа'],
    'RENI':  ['ренессанс страхование'],
    'DOMRF': ['дом.рф', 'дом рф'],
    'T':     ['т-технологии', 'тинькофф', 'tinkoff'],
    # ── Телеком и IT ─────────────────────────────────────────
    'MTSS':  ['мтс'],
    'RTKM':  ['ростелеком'],
    'VKCO':  ['вконтакте', 'vk', 'вк'],
    'YDEX':  ['яндекс', 'yandex'],
    'OZON':  ['озон', 'ozon'],
    'POSI':  ['группа позитив', 'positive technologies', 'позитив технологии'],
    'HEAD':  ['хэдхантер', 'headhunter', 'hh.ru'],
    'CNRU':  ['циан', 'cian'],
    # ── Ретейл ───────────────────────────────────────────────
    'X5':    ['x5', 'икс5', 'пятёрочка', 'перекрёсток', 'чижик', 'x5 retail'],
    'MGNT':  ['магнит'],
    'FIXP':  ['fix price', 'фикс прайс'],
    'MDMG':  ['мать и дитя', 'мать и дитя', 'md medical'],
    'GCHE':  ['черкизово'],
    'NKHP':  ['нкхп', 'новороссийский комбинат хлебопродуктов'],
    # ── Строительство ────────────────────────────────────────
    'PIKK':  ['пик', 'пик-специализированный застройщик'],
    'LSRG':  ['лср', 'группа лср'],
    'SMLT':  ['самолет', 'группа самолет'],
    # ── Энергетика ───────────────────────────────────────────
    'IRAO':  ['интерра', 'интер рао'],
    'MSNG':  ['мосэнерго'],
    'TGKA':  ['тгк-1'],
    'DVEC':  ['дэк', 'дальневосточная энергетическая компания'],
    # ── Транспорт и прочее ───────────────────────────────────
    'AFLT':  ['аэрофлот', 'aeroflot'],
    'FLOT':  ['совкомфлот'],
    'FESH':  ['двмп', 'fesco'],
    'AFKS':  ['акционерная финансовая корпорация система', 'афк система', 'акционерная корпорация система'],
    'PHOR':  ['фосагро', 'phosagro'],
    'KAZT':  ['казаньоргсинтез'],
    'LENT':  ['лента'],
    'AQUA':  ['инарктика', 'русская аквакультура'],
    'SFIN':  ['сафмар', 'sfin'],
    'TATN':  ['татнефть', 'tatneft'],
}

# Компилируем паттерны для скорости
COMPILED = {
    ticker: re.compile('|'.join(re.escape(kw) for kw in kws), re.IGNORECASE)
    for ticker, kws in COMPANY_KEYWORDS.items()
}

print(f'✓ Загружено {len(COMPANY_KEYWORDS)} тикеров')

In [ ]:
# ── ЯЧЕЙКА 3: Разметка ────────────────────────────────────────

df = pd.read_csv(INPUT_FILE, encoding='utf-8-sig')
df = df[df['scrape_status'] == 'ok'].copy()
df['text']  = df['text'].fillna('')
df['title'] = df['title'].fillna('')
print(f'Статей для разметки: {len(df)}')

rows = []
already_matched = {}  # для GAZP/SIBN: если SIBN сматчен, GAZP не добавляем

for idx, art in df.iterrows():
    text_lower  = art['text'].lower()
    title_lower = art['title'].lower()
    sibn_matched = False

    for ticker, pattern in COMPILED.items():
        # SIBN/GAZP логика
        if ticker == 'GAZP' and sibn_matched:
            continue

        body_count  = len(pattern.findall(text_lower))
        title_count = len(pattern.findall(title_lower))

        if body_count >= MIN_BODY_MENTIONS or title_count >= MIN_TITLE_MENTIONS:
            rows.append({
                'num':           art['num'],
                'ticker':        ticker,
                'title':         art['title'],
                'date':          art['date'],
                'media':         art['media'],
                'media_index':   art.get('media_index'),
                'visibility':    art.get('visibility'),
                'url_clean':     art['url_clean'],
                'domain':        art['domain'],
                'text':          art['text'],
                'body_mentions': body_count,
                'title_mentions': title_count,
            })
            if ticker == 'SIBN':
                sibn_matched = True

df_long = pd.DataFrame(rows)
print(f'\nРезультат: {len(df_long)} пар статья-тикер')
print(f'Уникальных статей сматчено: {df_long["num"].nunique()}')
print(f'\nТоп тикеров по числу статей:')
print(df_long['ticker'].value_counts().head(15).to_string())

df_long.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig',
               sep=';', lineterminator='\n')
print(f'\n✓ Сохранено: {OUTPUT_FILE}')